# 05 - Feature Engineering and Preprocessing

This notebook prepares raw customer features for machine learning while preventing data leakage.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

project_root = Path("..").resolve()
df = pd.read_csv(project_root / "dataset" / "processed" / "cleaned_telco.csv")
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Missing-Value Verification

Before feature construction, verify that the cleaned dataset contains no missing values. Even though the current cleaned data has no blanks, reusable pipelines include median imputation for numerical fields and most-frequent imputation for categorical fields.

In [2]:
missing_values = df.isna().sum()
missing_total = int(missing_values.sum())
print("Missing values in total:", missing_total)
print(missing_values[missing_values > 0] if missing_total else "No missing values found.")

Missing values in total: 0
No missing values found.


## Target and Feature Separation

`customerID` is an identifier and is removed because it should not help generalize to unseen customers. `Churn` is encoded as a binary target: 1 for Yes and 0 for No.

In [3]:
df = df.copy()
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
if df["Churn"].isna().any():
    raise ValueError("Unexpected Churn labels were found.")

X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"]

numerical_columns = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical columns:", numerical_columns)
print("Categorical columns:", categorical_columns)
print("X shape:", X.shape)
print("Target distribution:", y.value_counts().to_dict())

Numerical columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
X shape: (7043, 19)
Target distribution: {0: 5174, 1: 1869}


## Preprocessing Pipeline

Numerical features are standardized with `StandardScaler`.
Categorical features are converted to dummy indicators with `OneHotEncoder(handle_unknown='ignore')`.
`ColumnTransformer` applies the correct transformation to each feature type.

In [4]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessing_pipeline = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numerical_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

## Stratified Train-Test Split

A 20% test set is reserved before fitting preprocessing. Stratification preserves the class ratio approximately in both sets. A fixed random state makes the split reproducible.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Training churn ratio:", y_train.mean())
print("Test churn ratio:", y_test.mean())

Training shape: (5634, 19)
Test shape: (1409, 19)
Training churn ratio: 0.2653532126375577
Test churn ratio: 0.2654364797728886


## Fit Preprocessing and Verify Dimensions

The preprocessing pipeline is fitted only on training data. The test data is transformed afterward.

In [6]:
X_train_transformed = preprocessing_pipeline.fit_transform(X_train)
X_test_transformed = preprocessing_pipeline.transform(X_test)

print("Training transformed shape:", X_train_transformed.shape)
print("Test transformed shape:", X_test_transformed.shape)
print("Training and test dimensions match:", X_train_transformed.shape[1] == X_test_transformed.shape[1])

Training transformed shape: (5634, 45)
Test transformed shape: (1409, 45)
Training and test dimensions match: True


In [7]:
preprocessing_artifact = {
    "pipeline": preprocessing_pipeline,
    "numerical_columns": numerical_columns,
    "categorical_columns": categorical_columns,
    "target_mapping": {"Yes": 1, "No": 0}
}
joblib.dump(preprocessing_artifact, project_root / "models" / "churn_preprocessing.joblib")

split_artifact = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test,
    "X_train_transformed": X_train_transformed,
    "X_test_transformed": X_test_transformed
}
joblib.dump(split_artifact, project_root / "models" / "churn_train_test.joblib")

print("Saved preprocessing pipeline and train/test split artifacts.")

Saved preprocessing pipeline and train/test split artifacts.
